# Introducing Cloudera Lakehouse Optimizer (CLO)

In the previous labs, you experienced the "manual" side of Iceberg management—writing code to compact files and running procedures to prune metadata. While essential for performance, doing this at scale is a significant operational burden.

![image9.jpeg](assets/image9.jpeg)

**Cloudera Lakehouse Optimizer** is a fully managed, "set-and-forget" service designed to automate these tasks. By shifting from manual housekeeping to intelligent automation, you ensure your lakehouse remains fast and cost-effective without manual intervention.

---

### Why use Automated Optimization?

* **Intelligent Execution:** Instead of running on a simple timer, CLO monitors table health (file counts, metadata bloat, and scan costs) to trigger optimization only when necessary.
* **Operational Scale:** Manage thousands of tables across different environments from a single policy interface.
* **Performance Stability:** Prevents the "performance drift" that occurs as tables grow, maintaining the high-speed query performance you observed earlier.
* **Cost Efficiency:** Automatically reclaims storage by purging old snapshots and orphan files, directly reducing your cloud storage bill.


In this final section, you will explore how to configure an optimization policy to keep your data AI-ready and high-performing around the clock.

**Directions:**
* Run the stats check on the duplicate unoptised table again so that we have the baseline.

In [ ]:
# Enter your assigned user below
username = "user003"

from datetime import datetime
import cml.data_v1 as cmldata

SPARK_CONNECTION_NAME = "rvh-aw-dl"
conn_spark = cmldata.get_connection(SPARK_CONNECTION_NAME)
spark = conn_spark.get_spark_session()

print("Code block completed")

In [ ]:
%run -i assets/get_table_stats.py lakehouse_optimizer_$username sensor_telemetry_original

print("Code block completed")

## Step 1: Accessing the Lakehouse Optimizer

To begin automating our table maintenance, we need to navigate to the **Lakehouse Optimizer** within the Cloudera Management Console.

![image10.jpeg](assets/image10.jpeg)

**Directions:**
1.  From the left-hand navigation menu, click on **Lakehouse Optimizer**.
2.  Ensure your specific environment (e.g., `rvh-cdp-env`) is selected in the environment dropdown.
3.  Click the blue **Create Policy** button in the top right corner of the policies tab to start the configuration wizard.

---

## Step 2: Defining Policy General Information

In this step, we define the scope and identification for our optimization policy.

![image11.jpeg](assets/image11.jpeg)

**Directions:**
1.  **Select Policy Scope**: Choose **Namespace**. This allows us to target specific databases (namespaces) rather than applying the policy to the entire catalog.
2.  **Policy Name**: Enter a unique name for your policy, such as `user001_policy`. 
    > **Note:** In a shared environment, using your username helps distinguish your policy from others.
3.  **Description**: Provide a brief summary of the policy's purpose (e.g., *Housekeep for my Iceberg tables*).
4.  Click **Next** to proceed to namespace selection.

---

## Step 3: Namespace Associations

In this step, we tell the optimizer exactly which tables it should monitor. By selecting your specific database, any Iceberg table within it will automatically be evaluated for optimization.

![image12.jpeg](assets/image12.jpeg)

**Directions:**
1.  **Select Namespace**: Click the dropdown and select your database (e.g., `lakehouse_optimizer_user001`).
2.  **Select Tables**: 
    * Check the box next to your namespace to select all tables within it.
    * Alternatively, you can manually select `sensor_telemetry` and `sensor_telemetry_original` in the left pane.
3.  **Confirm Selection**: Verify that the tables appear in the **Selected Tables** pane on the right.
4.  Click **Next** to move to the Action configuration.

---

## Step 4: Configuring Policy Actions

This is where we define the "intelligence" of the policy—deciding when and how the optimizer should perform maintenance.

![image13.jpeg](assets/image13.jpeg)

**Directions:**
1.  **Table Maintenance Schedule**: Select **Schedule Based**. This ensures the optimizer evaluates your tables at a specific time.
2.  **Frequency & Time Zone**: 
    * Set the frequency to **Daily**.
    * Ensure the **Time Zone** matches your current local time (e.g., *GMT+01:00*).
    * Choose a time (Hour/Minutes) close to your current time if you want to see the policy trigger shortly after creation.
3.  **Automated Action**: Scroll down to the **Expire Snapshot** section and ensure the checkbox is enabled.
4.  **Retention Settings**:
    * **Expire Older Than**: Set this to **10** (seconds). In a real production environment, this would be days, but for this lab, we want immediate results.
    * **Retain Last**: Set this to **1**. This ensures we always keep at least the most recent "clean" version of the table while purging all the old, bloated metadata.
5.  Click **Next** to move to the final review.

---

## Step 5: Configuring Compaction Settings

In this sub-section of the Policy Actions, we configure the **Compaction** engine. This is the automated version of the `rewrite_data_files` procedure you ran manually in the previous lab.

![image14.jpeg](assets/image14.jpeg)

**Directions:**
1.  **Clean Expired Files**: Ensure this checkbox is **checked**. This tells the optimizer to physically delete the data files and manifests that are no longer needed after a snapshot expires.
2.  **Enable Compaction**: Check the box for **Compaction**. This enables the automatic merging of small files into larger, optimized ones.
3.  **Target File Size**: Set the Target File Size to **134217728** (which is 128 MB in bytes). This is the "Goldilocks" size for Iceberg data files—large enough to reduce metadata overhead but small enough for efficient distribution.
4.  **Thresholds**: Leave the other settings (Max Concurrent Rewrite, Minimum Input Files, etc.) at their default values. The Optimizer is smart enough to use these defaults to balance performance with cluster resources.
5.  Click **Next** to move to the final review screen.

---

## Step 6: Review and Finalize

Before the policy goes live, the Review screen provides a comprehensive overview of your automation strategy.

![image15.jpeg](assets/image15.jpeg)

**Directions:**
1.  **Verify Namespace Associations**: Confirm that your target namespaces and tables (e.g., `sensor_telemetry` and `sensor_telemetry_original`) are correctly listed.
2.  **Audit Policy Actions**:
    * **Schedule**: Confirm the execution time.
    * **Expire Snapshot**: Verify that "Clean Expired Files" is set to **true** and "Retain Last" is set to **1**.
    * **Compaction**: Ensure the target file size is set to **128MB** (134217728 bytes).
3.  **Finish**: Once you have verified all settings, click the **Finish** button at the bottom of the screen.

---

## Step 7: Monitoring Policy Execution

Once the policy is created, you can monitor its status and activity directly from the Lakehouse Optimizer dashboard.

![image16.jpeg](assets/image16.jpeg)

**Directions:**
1. **Verify Policy Status**: Look for your policy (e.g., `user001_policy`) in the list. A **green checkmark** in the Status column indicates the policy is active and healthy.
2. **Review Metrics**:
    * **Number of Tables**: Confirms how many tables are currently being managed by this policy (in this case, **3**).
    * **Number of Tasks**: Shows how many background maintenance tasks (Compaction or Snapshot Expiration) have been triggered.
3. **Last Execution**: Note the timestamp. This tells you exactly when the Optimizer last evaluated your tables and performed housekeeping.

---

## Step 8: Table-Level Observability

While the Policies tab shows you the "rules," the **Tables** tab provides a granular view of how those rules are being applied to your specific datasets.

![image17.jpeg](assets/image17.jpeg)

**Directions:**
1.  **Navigate to Tables**: Click on the **Tables** tab next to the Policies tab.
2.  **Filter by Namespace**: Use the namespace dropdown to select your database (e.g., `lakehouse_optimizer_user001`). 
3.  **Audit Individual Tables**: 
    * **Status**: A green checkmark confirms that each table is currently "Optimized" and compliant with your policy.
    * **Policies**: You can see exactly which policy is governing each table. If a table were part of multiple policies, they would all appear here.
    * **Last Execution**: This confirms the specific time each individual table was last processed. Note that even within the same policy, different tables might finish at slightly different times depending on their size and complexity.

---

## Step 9: Final Verification

Now that the **Cloudera Lakehouse Optimizer** has finished its first pass, let's verify the physical state of the table using our helper script.

1.  Run the following cell to collect the updated statistics for your optimized table:

In [ ]:
%run -i assets/get_table_stats.py lakehouse_optimizer_$username sensor_telemetry_original

print("Code block completed")

### Analyzing the Results

The output from our statistics script confirms that the **Cloudera Lakehouse Optimizer** has successfully transformed the table. 

<table style="margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr style="border-bottom: 2px solid #ddd;">
      <th style="text-align: left; padding: 8px;">Metric</th>
      <th style="text-align: left; padding: 8px;">Observation</th>
    </tr>
  </thead>
  <tbody>
    <tr style="border-bottom: 1px solid #eee;">
      <td style="padding: 8px;"><b>Total Files</b></td>
      <td style="padding: 8px;">Reduced from <b>750+</b> to exactly <b>30</b>.</td>
    </tr>
    <tr style="border-bottom: 1px solid #eee;">
      <td style="padding: 8px;"><b>Average File Size</b></td>
      <td style="padding: 8px;">Increased from <b>~1MB</b> to <b>~25MB</b> (A much healthier ratio for this dataset size).</td>
    </tr>
    <tr style="border-bottom: 1px solid #eee;">
      <td style="padding: 8px;"><b>Metadata Bloat</b></td>
      <td style="padding: 8px;">Notice that the <b>Snapshot count is now 1</b>. The optimizer successfully expired the hundreds of old snapshots we had from our initial "bloated" write.</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><b>Storage Efficiency</b></td>
      <td style="padding: 8px;">The metadata footprint is now less than <b>1MB</b> total.</td>
    </tr>
  </tbody>
</table>



### Why this matters
By reducing the file count by **96%**, we have eliminated the "Small File Syndrome" that was causing our query to take over 15 seconds. 

Because the optimizer is now active, this table will **never get bloated again**. As new data flows in, CLO will continue to merge small files and prune old metadata in the background, ensuring that your 1.88-second query performance remains consistent over time.

**Mission Accomplished! You have successfully optimized a Lakehouse table from end-to-end.**

**Final Table State Comparison:**

<table style="margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr style="border-bottom: 2px solid #ddd;">
      <th style="text-align: left; padding: 8px;">Feature</th>
      <th style="text-align: center; padding: 8px;">Bloated (Before)</th>
      <th style="text-align: center; padding: 8px;">Optimized (After)</th>
    </tr>
  </thead>
  <tbody>
    <tr style="border-bottom: 1px solid #eee;">
      <td style="padding: 8px;"><b>File Layout</b></td>
      <td style="text-align: center; padding: 8px;">🧊🧊🧊...🧊 (750+)</td>
      <td style="text-align: center; padding: 8px;">🧊🧊🧊 (30)</td>
    </tr>
    <tr style="border-bottom: 1px solid #eee;">
      <td style="padding: 8px;"><b>Query Speed</b></td>
      <td style="text-align: center; padding: 8px;">🐢 15.67s</td>
      <td style="text-align: center; padding: 8px;">🚀 1.88s</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><b>Status</b></td>
      <td style="text-align: center; padding: 8px;">⚠️ At Risk</td>
      <td style="text-align: center; padding: 8px;">✅ AI-Ready</td>
    </tr>
  </tbody>
</table>

## &#x1F600; Well done - You've completed this section of the labs!